In [ ]:
# Celda 1 — Setup
# Añadir la raíz del proyecto al path para poder importar los módulos del juego.
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# SDL_VIDEODRIVER=dummy evita que pymunk/pygame intenten abrir una ventana.
# Es indispensable en un notebook headless.
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

import pygame
pygame.init()

import pymunk
import numpy as np
import matplotlib.pyplot as plt

from game.terrain import Terrain

print('Imports OK')

In [ ]:
# Celda 2 — Muestrear height_at() con dos semillas distintas
#
# Creamos dos terrenos con seeds diferentes y evaluamos height_at(x)
# para x en [0, 5000]. Usamos Space vacíos solo para cumplir la interfaz;
# no necesitamos física aquí.

SEED_A = 42
SEED_B = 7

space_a = pymunk.Space()
space_b = pymunk.Space()

terrain_a = Terrain(space=space_a, seed=SEED_A)
terrain_b = Terrain(space=space_b, seed=SEED_B)

xs = np.linspace(0, 5000, 2000)  # 2000 puntos → resolución 2.5 px

# height_at() devuelve y en coordenadas pygame (Y↓).
# Negamos para que colinas aparezcan como picos en la gráfica.
ys_a = np.array([-terrain_a.height_at(float(x)) for x in xs])
ys_b = np.array([-terrain_b.height_at(float(x)) for x in xs])

print(f'Seed {SEED_A} — rango y: [{ys_a.min():.1f}, {ys_a.max():.1f}] px')
print(f'Seed {SEED_B} — rango y: [{ys_b.min():.1f}, {ys_b.max():.1f}] px')

In [ ]:
# Celda 3 — Graficar perfiles
#
# Qué observar:
#   - Zona plana hasta x=400 en ambas semillas (spawn sin pendiente).
#   - Dificultad creciente: las colinas se vuelven más pronunciadas a la derecha.
#   - Perfiles distintos entre seeds (fases distintas).
#   - Meseta de dificultad máxima a partir de x ≈ 3400 (400 + 3000).

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, ys, seed, color in zip(
    axes,
    [ys_a, ys_b],
    [SEED_A, SEED_B],
    ['steelblue', 'tomato'],
):
    ax.fill_between(xs, ys.min() - 20, ys, alpha=0.3, color=color)
    ax.plot(xs, ys, color=color, linewidth=1.2, label=f'seed={seed}')
    ax.axvline(400,  color='gray', linestyle='--', linewidth=0.8, label='fin zona plana (x=400)')
    ax.axvline(3400, color='gray', linestyle=':',  linewidth=0.8, label='dificultad máxima (x=3400)')
    ax.set_ylabel('Altura (px, invertido)')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Posición x (px)')
fig.suptitle('Perfiles de terreno procedural — sub-fase 8.1', fontsize=13)
plt.tight_layout()
plt.savefig('terrain_profile.png', dpi=120)
plt.show()
print('Guardado: experiments/terrain_profile.png')

In [ ]:
# Celda 4 — Verificar determinismo
#
# El mismo seed debe producir exactamente el mismo perfil en dos instancias
# distintas. Esta celda confirma que no hay estado global contaminando el RNG.

space_c = pymunk.Space()
terrain_c = Terrain(space=space_c, seed=SEED_A)  # misma seed que terrain_a

ys_c = np.array([-terrain_c.height_at(float(x)) for x in xs])

max_diff = np.max(np.abs(ys_a - ys_c))
print(f'Diferencia máxima entre dos instancias con seed={SEED_A}: {max_diff:.10f} px')
assert max_diff == 0.0, 'ERROR: el terreno NO es determinista con la misma seed!'
print('Determinismo verificado: mismo seed → perfil idéntico.')